In [36]:
import os
import numpy as np
from PIL import Image

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [37]:
# Création des variables label et idx (id au pluriel)
X = ["glioma", "meningioma", "notumor", "pituitary"]
y = {name: id for id, name in enumerate(X)}

# Dimension des images 256 par 256
imgs_size = (256, 256)

# Fonction permettant de donner un id de classe à chaques photos
def listlabels(root_dir):
    filepaths = []
    labels = []

    for k in X:
        class_dossier = os.path.join(root_dir, k)
        if not os.path.isdir(class_dossier):
            raise FileNotFoundError(f"Dossier attendu introuvable : {class_dossier}")

        for l in os.listdir(class_dossier):
            if l.lower().endswith((".jpg", ".jpeg", ".png")):
                filepaths.append(os.path.join(class_dossier, l))    # 
                labels.append(y[k])  # Ajout de l'id de la classe de chaques photo dans labels

    return filepaths, np.array(labels)

# Fonction qui permet de récuper toutes les images et de les normaliser
def load_images(filepaths, img_size=imgs_size):
    images = []

    for k in filepaths:
        img = Image.open(k)
        img = img.convert("L") # Convertion en Noir et blanc grayscale True
        img = img.resize(img_size) # Resize les images pour les avoir toutes dans le même format
        images.append(np.asarray(img, dtype=np.float32) / 255.0)    # Ajout de la nouvelle image dans images

    return np.stack(images)



train_paths, y_train = listlabels("Training")
test_paths, y_test = listlabels("Testing")

print(f"Train : {len(train_paths)} images")
print(f"Test  : {len(test_paths)} images")

for name, id in y.items():
    print(f"{name} -> train: {np.sum(y_train == id)} | test: {np.sum(y_test == id)}")

X_train = load_images(train_paths)
X_test = load_images(test_paths)

X_train_plat = X_train.reshape(X_train.shape[0], -1)
X_test_plat = X_test.reshape(X_test.shape[0], -1)

print(f"X_train : {X_train.shape}")
print(f"X_test : {X_test.shape}")   

print(X_train_plat.shape)
print(X_test_plat.shape)

Train : 5600 images
Test  : 1600 images
glioma -> train: 1400 | test: 400
meningioma -> train: 1400 | test: 400
notumor -> train: 1400 | test: 400
pituitary -> train: 1400 | test: 400
X_train : (5600, 256, 256)
X_test : (1600, 256, 256)
(5600, 65536)
(1600, 65536)


 # Modèle KNN


In [40]:
knn = KNeighborsClassifier(n_neighbors=9)

knn.fit(X_train_plat, y_train)

y_pred = knn.predict(X_test_plat)


accuracy = accuracy_score(y_test, y_pred)

print("Accuracy :", accuracy)

def load_one_image(file, img_size=imgs_size):
    images = []

    img = Image.open(file)
    img = img.convert("L") # Convertion en Noir et blanc grayscale True
    img = img.resize(img_size) # Resize les images pour les avoir toutes dans le même format
    images.append(np.asarray(img, dtype=np.float32) / 255.0)    # Ajout de la nouvelle image dans images

    return np.stack(images)

nouvelle_image = load_one_image("Testing/glioma/Te-gl_18.jpg")

nouvelle_image_plat = nouvelle_image.reshape(nouvelle_image.shape[0], -1)

prediction = knn.predict(nouvelle_image_plat)

print(X[prediction[0]])

# X = CSVIris[["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]]
# y = CSVIris["Species"]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# neigh = KNeighborsClassifier(n_neighbors=16)
# neigh.fit(X_train, y_train)
# y_pred = neigh.predict(X_test)

# print("Score train:", neigh.score(X_train, y_train))
# print("Score test:", neigh.score(X_test, y_test))

# nouvelles_fleurs = pd.DataFrame({
#     "SepalLengthCm": [5.2, 6.78, 6.65],
#     "SepalWidthCm":  [3.9, 4.96, 4.3],
#     "PetalLengthCm": [2.35, 6.0, 3.0],
#     "PetalWidthCm":  [1.17, 4.485, 2.21]
# })

# predictions = neigh.predict(nouvelles_fleurs)
# print(predictions)
# probas = neigh.predict_proba(nouvelles_fleurs)
# print(probas)

Accuracy : 0.7325
glioma
